# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs, referencing each by their `@id` field.

In [ ]:
# List all record sets by their @id
def print_record_sets(ds):
    print("Available Record Sets (@id and name):")
    for rs in ds.record_sets:
        print(f"  @id: {rs.id}, name: {rs.name}")

print_record_sets(dataset)

# As an example, enumerate fields and columns for each record set
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
            if hasattr(field, 'columns'):
                print("      Columns:")
                for col in getattr(field, 'columns', []):
                    if hasattr(col, 'id') and hasattr(col, 'name'):
                        print(f"        @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> We'll load **all record sets** for demonstration. Update the lists below as needed using their `@id` values from above.

In [ ]:
# Define the list of record set @ids (update this to match your data)
record_sets_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Loaded DataFrame for {record_set_id} with columns:", df.columns.tolist())
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Let's select a record set and a numeric field (by `@id`) for processing.

***Please update `chosen_record_set_id` and `numeric_field_id` below based on the overview step.***

In [ ]:
# Example record set and numeric field (replace with your actual @ids)
# For demonstration, we'll use the first record set with a numeric column
chosen_record_set_id = None
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    for field in getattr(rs, 'fields', []):
        dt = getattr(field, 'data_type', None)
        if dt in ('Float', 'Integer', 'Number'):
            chosen_record_set_id = rs.id
            numeric_field_id = field.id
            break
    if chosen_record_set_id:
        break
# Try to get group field (if available)
if chosen_record_set_id is not None:
    colnames = dataframes[chosen_record_set_id].columns.tolist()
    # Try typical group columns
    for group_try in ['group', 'category', 'Sample', 'sample', 'SampleType', 'Type', 'type']:
        if group_try in colnames:
            group_field_id = group_try
            break
if chosen_record_set_id is None:
    raise ValueError("No numeric field found.")
print(f"Chosen record set: {chosen_record_set_id}")
print(f"Numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Group field: {group_field_id}")

# EDA: filtering, normalization, grouping
df = dataframes[chosen_record_set_id]
if numeric_field_id in df.columns:
    # Try cast to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.25)  # Use 25th percentile for demo
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head(3))

    normalized_field = numeric_field_id + "_normalized"
    filtered_df[normalized_field] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_field]].head(3))

    # Group by another field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Column '{numeric_field_id}' not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()


## 6. Conclusion
In this notebook, you've learned how to:

- Load dataset metadata and records using `mlcroissant` from a Croissant schema URL.
- Discover available record sets, fields, and reference entities by their `@id` fields.
- Extract, filter, normalize, and aggregate data programmatically.
- Visualize numeric distributions and grouped statistics for initial exploratory data analysis.

Proceed by drilling into domain-specific exploration, advanced visualizations, or downstream modeling tasks for more scientific insight.